# Project 03 — Medical Image Anomaly Detector
## Phase 1 Training (Binary: NORMAL vs PNEUMONIA)

This notebook runs **Phase 1 only** (head training, backbone frozen) on a Colab T4 GPU.

**Gate to clear before moving on:** validation mean AUC > 0.70

Steps below: check GPU → upload project code → install deps → download dataset → sanity-check the pipeline with pytest → train Phase 1 → inspect results.

Runtime: **Runtime → Change runtime type → T4 GPU** before running anything below.

## 1. Confirm GPU is available

In [ ]:
!nvidia-smi

## 2. Upload the project code

Run the cell below, then in the file picker select **`chest-xray-detector.zip`**
(the project zip from this conversation — it contains `src/`, `tests/`,
`run_training.py`, and `requirements.txt`).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select chest-xray-detector.zip


In [ ]:
import zipfile, os

zip_name = [f for f in uploaded.keys() if f.endswith('.zip')][0]
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content/')

# The zip may extract as /content/chest-xray-detector/... — locate it
for root, dirs, _ in os.walk('/content'):
    if 'src' in dirs and 'run_training.py' in os.listdir(root):
        PROJECT_DIR = root
        break

%cd $PROJECT_DIR
print(f"Project directory: {PROJECT_DIR}")
!ls


## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Google Drive (for checkpoint persistence)

Colab sessions disconnect without warning — checkpoints go to Drive so a
disconnect during a multi-hour run doesn't lose progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/checkpoints/chest_xray'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")


## 5. Set up Kaggle API access

1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) → **API** → **Create New Token**
   This downloads a `kaggle.json` file.
2. Run the cell below and upload that `kaggle.json` when prompted.

In [ ]:
from google.colab import files
print("Upload your kaggle.json file:")
kaggle_uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle credentials installed.")


## 6. Download the dataset (~2GB, Kaggle Chest X-Ray Images Pneumonia)

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --unzip


## 7. Locate the real `train/` folder

The Kaggle zip sometimes extracts with an extra nested `chest_xray/chest_xray/`
folder depending on download method. This cell finds wherever `train/`
(containing NORMAL/PNEUMONIA subfolders) actually landed, instead of
hardcoding a path that might be wrong.

In [ ]:
import os

def find_data_root(search_root='/content/data'):
    for root, dirs, _ in os.walk(search_root):
        if 'train' in dirs:
            candidate = os.path.join(root, 'train')
            subdirs = os.listdir(candidate)
            if 'NORMAL' in subdirs and 'PNEUMONIA' in subdirs:
                return root
    raise FileNotFoundError(
        f"Could not find a train/ folder with NORMAL/PNEUMONIA subfolders under {search_root}. "
        f"Run `!find {search_root} -maxdepth 3` to inspect the extracted structure manually."
    )

DATA_ROOT = find_data_root()
print(f"DATA_ROOT = {DATA_ROOT}")
!ls "$DATA_ROOT"
!echo "---"
!ls "$DATA_ROOT/train"


## 8. Sanity-check the pipeline before burning GPU hours

Runs the full test suite from Weeks 1–2 against the real environment
(confirms torch/torchvision versions, EfficientNetB3 download, etc. all
work here — not just in the original dev sandbox).

In [ ]:
!python -m pytest tests/ -q

## 9. (Optional) Weights & Biases login

Skip this cell if you'd rather not track the run — pass `--no-wandb` in
step 11 instead.

In [ ]:
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize when prompted


## 10. Quick smoke test — 1 epoch, no pretrained download, no wandb

Confirms the full train_epoch/val_epoch loop runs end-to-end on REAL data
before committing GPU time to a full Phase 1 run. Takes ~1-2 minutes.

In [ ]:
!python run_training.py \
    --data-root "$DATA_ROOT" \
    --checkpoint-dir /content/checkpoints_smoketest \
    --epochs1 1 \
    --epochs2 0 \
    --no-wandb \
    --no-pretrained


## 11. Run Phase 1 (full — 5 epochs, head only, backbone frozen)

`--epochs2 0` skips Phase 2 entirely for now — we're only clearing the
Phase 1 gate (val AUC > 0.70) before deciding whether to continue.

Expect roughly 1-3 minutes/epoch on a T4 for this dataset size (~4,200
train images), so this should take well under 15 minutes total.

In [ ]:
!python run_training.py \
    --data-root "$DATA_ROOT" \
    --checkpoint-dir "$CHECKPOINT_DIR" \
    --epochs1 5 \
    --epochs2 0


## 12. Check the result against the gate

**Gate: validation mean AUC > 0.70**

If you cleared it, you're ready to move to Phase 2 (fine-tuning) and then
Week 3 (`evaluate.py` — full test-set AUC, F1, confusion matrix).

If you didn't clear it, common causes worth checking first: dataset really
loaded the full ~4,200 train images (not a tiny subset), `pos_weight` looks
sane (~0.35), and the loss was actually decreasing across epochs 1→5 in the
logs above.

In [ ]:
import torch

ckpt = torch.load(f"{CHECKPOINT_DIR}/best_model.pth", map_location='cpu')
print(f"Best epoch: {ckpt['epoch'] + 1}")
print(f"Best val AUC: {ckpt['best_auc']:.4f}")
print()
if ckpt['best_auc'] > 0.70:
    print("✓ GATE CLEARED — ready for Phase 2 fine-tuning")
else:
    print("✗ Gate not yet cleared — review training curves before proceeding")


## 13. (When ready) Run Phase 2 — fine-tuning

Resumes is not yet wired up automatically here (that lands with
`evaluate.py` in Week 3), so this re-runs Phase 1 + Phase 2 together. Phase 1
is fast (head-only), so this is fine for now.

In [ ]:
!python run_training.py \
    --data-root "$DATA_ROOT" \
    --checkpoint-dir "$CHECKPOINT_DIR" \
    --epochs1 5 \
    --epochs2 10
